# Phase 3 — Feature Engineering

Thin wrapper around `src/features.py` — the reusable logic lives there (also used directly by `train_ml.py`/`train_dl.py`), so this notebook is for *inspecting* the result, not duplicating the code.

Run `python -m src.data_prep` first so `data/processed/clean_v1.csv` exists.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from src import config
from src.features import build_feature_matrix, time_based_split

In [ ]:
clean = pd.read_csv(config.CLEAN_CSV_PATH, parse_dates=[config.DATE_COL])
clean.head()

## Build the feature matrix

In [ ]:
features = build_feature_matrix(clean)
features.shape

In [ ]:
features.head()

## Sanity-check for leakage

Every `lag_*` / `roll_mean_*` / `roll_std_*` column for a given row should only ever reflect earlier dates for that `mode`. Spot-check one row by hand:

In [ ]:
row = features[features['mode'] == 'total'].iloc[50]
row[['service_date', 'rides', 'lag_1', 'lag_2', 'lag_7',
     'roll_mean_7', 'roll_mean_14', 'roll_mean_28']]

## Save

Same file `train_ml.py` / `train_dl.py` / the R baselines (indirectly, via `clean_v1.csv`) all read from.

In [ ]:
config.FEATURES_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(config.FEATURES_CSV_PATH, index=False)
print(f'Saved {len(features):,} rows -> {config.FEATURES_CSV_PATH}')

## Confirm the split lines up with what Phase 4 (R) will use

In [ ]:
train_df, val_df, test_df = time_based_split(features)
print('train:', train_df[config.DATE_COL].min(), '->', train_df[config.DATE_COL].max())
print('val:  ', val_df[config.DATE_COL].min(), '->', val_df[config.DATE_COL].max())
print('test: ', test_df[config.DATE_COL].min(), '->', test_df[config.DATE_COL].max())